# Kafi

Welcome to the documentation the *Kafi* part of *Kafi Streams*.

Kafi is a shell-like Kafka interface for writing producers, consumers, or doing administrative tasks like backups.

Kafi is fun to use either in the interactive Python interpreter, inside your Python (micro-)service code, and - it's the ideal tool for interacting with Kafka in your Jupyter notebooks.

Kafi supports two main modes:
* *Real Kafka*
  * Kafka API via [confluent_kafka](https://github.com/confluentinc/confluent-kafka-python)
  * Kafka via [Confluent REST Proxy API](https://docs.confluent.io/platform/current/kafka-rest/api.html)
* *Emulated Kafka/files*
  * local file system
  * S3
  * Azure Blob Storage

Kafi treats *emulated Kafka* in the same way as *real Kafka*. The only difference is that there is no need to run an additional Kafka cluster. Typical use cases are downloading snapshots of Kafka topics or doing backups.

Kafi fully supports the Schema Registry API (supporting Avro, Protobuf and JSONSchema).

## Table of Contents

**Basics**
- [Installation](#installation)
- [Basic Configuration](#basic-configuration)
- [Use Cases](#use-cases)

**Details**
- [Full Configuration](#full-configuration)
- [More on Producing Messages](#more-on-producing-messages)
- [More on Consuming Messages](#more-on-consuming-messages)
- [Architecture](#architecture)

---
## Installation

Kafi is on PyPI. Run the cell below to install it:

In [ ]:
!pip install kafi

---
Kafi is configured using YAML files.

As an example, here is a YAML file for a local Kafka installation including Schema Registry:

```yaml
kafka:
  bootstrap.servers: localhost:9092

schema_registry:
  schema.registry.url: http://localhost:8081
```

And this is a YAML file for emulated Kafka (on local disk) in the /tmp-directory:

```yaml
local:
  root.dir: /tmp
```

Kafi is looking for these YAML files in:
1. the current directory (`.`) or the directory set in `KAFI_HOME` (if set)
2. the `configs/<storage type>/<storage config>` sub-directory of `.` or `KAFI_HOME` (if set). Here, `storage_type` is either `azblobs`, `clusters`, `locals`, `restproxies` or `s3s`. `storage_config` is your configuration file (in Kafi, a connection to one of its back-ends is called *storage*) 

Within Kafi, you can refer to these files by their name without the `.yml` or `.yaml` suffix, e.g. `local` for `local.yaml` (or also for `local.yml`).

The YAML files can also contain environment variables, e.g.:
```yaml
kafka:
  bootstrap.servers: ${KAFI_KAFKA_SERVER}
  security.protocol: SASL_SSL
  sasl.mechanisms: PLAIN
  sasl.username: ${KAFI_KAFKA_USERNAME}
  sasl.password: ${KAFI_KAFKA_PASSWORD}
  
schema_registry:
  schema.registry.url: ${KAFI_SCHEMA_REGISTRY_URL}
  basic.auth.credentials.source: USER_INFO
  basic.auth.user.info: ${KAFI_SCHEMA_REGISTRY_USER_INFO}
```
This is useful for CI/CD pipelines where you use environment variables for injecting secrets from a separate secret store.

We provide example YAML files in this GitHub repository under `configs`:
* Real Kafka
  * Kafka API:
    * Local Kafka installation: `clusters/local.yaml`
    * Confluent Cloud: `clusters/ccloud.yaml`
    * Redpanda: `clusters/redpanda.yaml`
  * Kafka REST Proxy API:
    * Local Kafka/REST Proxy installation: `restproxies/local.yaml`
* Emulated Kafka/files
  * local file system: `locals/local.yaml`
  * S3: `s3s/local.yaml`
  * Azure Blob Storage: `azureblobs/local.yaml`

You can find more details on configuring Kafi [here](#full-configuration).


---
## Use Cases

### 1. An Alternative to the Existing CLI Tools

Kafi can act as an alternative to the existing Kafka CLI tools. The connection to your Kafka cluster is encapsulated in the `Cluster` object — you don't need to repeat connection details for each command.

First, make sure that you have a local Kafka running on localhost/port 9092. Then start by importing Kafi and creating a `Cluster` object:

In [1]:
import os, sys
os.environ["KAFI_HOME"] = ".."
sys.path.insert(1, "..")

from kafi.kafi import *
c = Cluster("local")

#### List Topics

Kafi equivalent of `kafka-topics --bootstrap-server localhost:9092 --list`:

In [2]:
c.ls()

['mirrored_topic_json',
 'topic_avro',
 'topic_avro_json_copy',
 'topic_json',
 'topic_json_avro_copy',
 'topic_json_copy',
 'topic_json_flatmapped',
 'topic_json_from_df',
 'topic_json_from_parquet',
 'topic_json_mapped',
 'topic_jsonschema',
 'topic_protobuf',
 'topic_protobuf_json_mapped']

#### Create Topics

Kafi equivalent of `kafka-topics --bootstrap-server localhost:9092 --topic topic_json --create`:

In [ ]:
c.touch("topic_json")

#### Produce Messages

Producing messages works similarly to writing to a file in Python:

In [ ]:
p = c.producer("topic_json")
p.produce({"bla": 123}, key="123")
p.produce({"bla": 456}, key="456")
p.produce({"bla": 789}, key="789")
p.close()

#### Consume Messages

In [ ]:
c.cat("topic_json")

#### Produce Messages Using a Schema

##### Avro

In [ ]:
t = "topic_avro"
s = """
{
    "type": "record",
    "name": "myrecord",
    "fields": [
        {
            "name": "bla",
            "type": "int"
        }
    ]
}
"""
p = c.producer(t, value_type="avro", value_schema=s)
p.produce({"bla": 123}, key="123")
p.produce({"bla": 456}, key="456")
p.produce({"bla": 789}, key="789")
p.close()

##### Protobuf

In [ ]:
t = "topic_protobuf"
s = """
message the_value {
    required int32 bla = 1;
}
"""
p = c.producer(t, value_type="protobuf", value_schema=s)
p.produce({"bla": 123}, key="123")
p.produce({"bla": 456}, key="456")
p.produce({"bla": 789}, key="789")
p.close()

##### JSONSchema

In [ ]:
t = "topic_jsonschema"
s = """
{
    "$schema": "http://json-schema.org/draft-07/schema#",
    "type": "object",
    "title": "myrecord",
    "properties": {
      "bla": {
        "type": "integer"
      }
    },
    "required": ["bla"],
    "additionalProperties": false
  }
"""
p = c.producer(t, value_type="jsonschema", value_schema=s)
p.produce({"bla": 123}, key="123")
p.produce({"bla": 456}, key="456")
p.produce({"bla": 789}, key="789")
p.close()

#### Search Messages

Use `grep` to search for messages matching a regex pattern:

In [ ]:
c.grep("topic_avro", ".*456.*", value_type="avro")

#### Supported Serialization/Deserialization Types

| Type | Description |
|------|-------------|
| `bytes` | Pure bytes |
| `str` | String *(default for keys)* |
| `json` | Pure JSON *(default for values)* |
| `avro` | Avro *(requires Schema Registry)* |
| `protobuf` / `pb` | Protobuf *(requires Schema Registry)* |
| `jsonschema` / `json_sr` | JSONSchema *(requires Schema Registry)* |

Set types via:
- `key_type` / `key_schema` / `key_schema_id`
- `value_type` / `value_schema` / `value_schema_id`
- `type` — same type for both key and value

---
### 2. A Convenient Tool for Schema Registry Administration

#### Get Subjects

In [ ]:
c.sls()

#### Delete a Subject

First, soft-delete:

In [ ]:
c.srm("topic_avro-value")

List subjects (soft-deleted subjects are hidden by default):

In [ ]:
c.sls()

Include soft-deleted subjects:

In [ ]:
c.sls(deleted=True)

Hard-delete the subject permanently:

In [ ]:
c.srm("topic_avro-value", permanent=True)

Verify that it's gone:

In [ ]:
c.sls(deleted=True)

#### Get the Latest Version of a Subject

In [ ]:
c.get_latest_version("topic_jsonschema-value")

---
### 3. Simple Stateless Stream Processing

#### Copy Topics

Simple copy:

In [ ]:
c.cp("topic_json", c, "topic_json_copy")

Convert Protobuf topic to pure JSON:

In [ ]:
c.cp("topic_protobuf", c, "topic_avro_json_copy", source_value_type="protobuf")

Copy a pure JSON topic to an Avro topic:

In [ ]:
s = """
{
    "type": "record",
    "name": "myrecord",
    "fields": [
        {
            "name": "bla",
            "type": "int"
        }
    ]
}
"""
c.cp("topic_json", c, "topic_json_avro_copy", target_value_type="avro", target_value_schema=s)

#### Copy + Map (=Map_To)

Use a *single message transform* via `map_fun`:

In [ ]:
def plus_42(x):
    x["value"]["bla"] += 42
    return x

c.cp("topic_json", c, "topic_json_mapped", map_fun=plus_42)

Check the result:

In [ ]:
c.cat("topic_json_mapped")

Map also works seamlessly with schemas:

In [ ]:
c.cp("topic_protobuf", c, "topic_protobuf_json_mapped", map_fun=plus_42, source_value_type="protobuf")

#### Copy + FlatMap (=FlatMap_To)

Use `flatmap_fun` for filtering or exploding messages:

In [ ]:
def filter_out_456(x):
    if x["value"]["bla"] == 456:
        return [x]
    else:
        return []

c.cp("topic_json", c, "topic_json_flatmapped", flatmap_fun=filter_out_456)

#### Like A MirrorMaker

Copy topics across clusters — input and output can be on any cluster:

In [ ]:
c1 = Cluster("local")
c2 = Cluster("local")
c1.cp("topic_json", c2, "mirrored_topic_json")

---
### 4. A Backup Tool

#### Backing Up a Topic to Local Disk

We use `type="bytes"` for a 1:1 carbon copy (no deserialization/serialization). Kafi's Kafka emulation preserves all metadata (keys, values, headers, timestamps).

In [ ]:
c = Cluster("local")
l = Local("local")
c.cp("topic_json", l, "backupped_topic_json", type="bytes")
!ls /tmp/topics/backupped_topic_json/partitions/*

#### Restoring a Backed-up Topic to Kafka

In [ ]:
l.cp("backupped_topic_json", c, "topic_json", type="bytes")

#### Backing Up a Topic to S3

For this example to work, you need to configure `s3` correctly first (see [Full Configuration](#full-configuration))

In [ ]:
c.cp("my_topic", s3, "my_topic_backup", type="bytes")

---
### 5. A Bridge from Kafka to Files

Kafi integrates with Pandas and supports these file formats: CSV, Feather, JSON, ORC, Parquet, Excel, XML.

#### Get a Snapshot of a Topic as a Pandas Dataframe

In [ ]:
df = c.topic_to_df("topic_protobuf", value_type="protobuf")
df

#### Write a Pandas Dataframe to a Kafka Topic

In [ ]:
c.df_to_topic(df, "topic_json_from_df")
c.cat("topic_json_from_df")

#### Get a Snapshot of a Topic as an Excel File

In [ ]:
l = Local("local")
c.topic_to_file("topic_json", l, "topic_json.xlsx")

#### Get a Snapshot of a Topic as a Parquet File

In [ ]:
l = Local("local")
c.topic_to_file("topic_json", l, "topic_json.parquet")

#### Bring a Parquet File Back to Kafka

In [ ]:
l = Local("local")
l.file_to_topic("topic_json.parquet", c, "topic_json_from_parquet")

---
### 6. A Powerful Debug Tool

#### Check for Missing Magic Byte

Find messages that were not properly serialized (magic byte 0 is missing):

In [ ]:
c.filter("topic_protobuf", type="bytes", filter_fun=lambda x: x["value"][0] != 0)

#### Delete Records

Delete the first 100 messages of a topic:

In [ ]:
t = "topic_jsonschema"
print(c.l(t))
c.delete_records({t: {0: 2}})
c.l(t)

#### Get topic watermarks


In [ ]:
c.watermarks("topic_jsonschema")

#### Collect All Schemas Used in a Topic

Collect all schema IDs in use and print the corresponding schemas from the Schema Registry:

In [ ]:
def collect_ids(acc, x):
    id = s_id(x["value"])
    acc.add(id)
    return acc

(ids, _) = c.foldl("topic_jsonschema", collect_ids, set(), type="bytes")

for id in ids:
    print(id)

---
## Full Configuration

In Kafi one configuration file corresponds to a "connection" to a so-called *storage* (Kafka API, Kafka REST Proxy API, Local file system, S3 and Azure Blob Storage). Each storage has one section that only makes sense for itself:

* Kafka API: `kafka`
* Kafka REST Proxy API: `rest_proxy`
* Local file system: `local`
* S3: `s3`
* Azure Blob Storage: `azure_blob`

In addition, the storages can all have one or two of the following sections:
* `schema_registry` (Schema Registry configuration)
* `kafi` (additional configuration items)

Please also have a look at the example YAML files in the GitHub repo for further illustration.

### General

The following configuration items are shared across all *storages* (defaults in brackets):

* `schema_registry`
  * `schema.registry.url`
  * `basic.auth.credentials.source`
  * `basic.auth.user.info`

* `kafi`
  * `progress.num.messages` (`1000`)
  * `consume.batch.size` (`1000`)
  * `produce.batch.size` (`1000`)
  * `verbose` (`1` if run in the interactive Python interpreter, `0` if not)
  * `auto.offset.reset` (`earliest`)
  * `consumer.group.prefix` (`""`)
  * `enable.auto.commit` (`false`)
  * `commit.after.processing` (`true`)
  * `key.type` (`str`)
  * `value.type` (`json`)

### Real Kafka

#### Kafka API

* `kafka`
  * `bootstrap.servers`
  * `security.protocol`
  * `sasl.mechanisms`
  * `sasl.username`
  * `sasl.password`
  * `log_level`(`3` if run in the interactive Python interpreter, `6` if not))
  * etc. [librdkafka configuration](https://github.com/confluentinc/librdkafka/blob/master/CONFIGURATION.md)

* `kafi`
  * `flush.timeout` (`-1.0`)
  * `retention.ms` (`604800000`)
  * `consume.timeout` (`5.0`)
  * `session.timeout.ms` (`45000`)

#### Kafka REST Proxy API

* `rest_proxy`:
  * `rest.proxy.url`
  * `basic.auth.user.info`

* `kafi`:
  * `fetch.min.bytes` (`-1`)
  * `consumer.request.timeout.ms` (`1000`)
  * `consume.num.attempts` (`3`)
  * `requests.num.retries` (`10`)

### Kafka Emulation/files

#### Local File System

* `local`:
  * `root.dir` (`.`)

#### S3

* `s3`:
  * `endpoint`
  * `access.key`
  * `secret.key`
  * `bucket.name` (`test`)
  * `root.dir` (`""`)

#### Azure Blob Storage

* `azure_blob`:
  * `connection.string`
  * `container.name` (`test`)
  * `root.dir` (`""`)


---
## More on Producing Messages

To streamline its syntax, Kafi employs a number of defaults/assumptions. All of them can of course be overridden.

Look at the following code from above:


In [ ]:
p = c.producer("topic_json")
p.produce({"bla": 123}, key="123")
p.produce({"bla": 456}, key="456")
p.produce({"bla": 789}, key="789")
p.close()

Kafi uses the following defaults/assumptions here. First, for setting up the producer object:
* The maximum batch size for producing is set to the corresponding value `produce.batch.size` in the `kafi` section of the configuration file, e.g. `1000` in `clusters/local.yaml`.
* The flush timeout for `flush` calls to the Kafka API is set to the corresponding value `flush.timeout` in the `kafi` section of the configuration file, e.g. `-1.0` in `clusters/local.yaml`.
* The default key type is set to the corresponding value `key.type` in the `kafi` section of the configuration file, e.g. `str` in `clusters/local.yaml`. It can also be overridden with the `key_type` kwargs parameter.
* The default value type is set to the corresponding value `value.type` in the `kafi` section of the configuration file, e.g. `json` in `clusters/local.yaml`. It can also be overridden with the `value_type` kwargs parameter.
* No delivery callback function is called. This can be overridden with the `delivery_fun` kwargs parameter.

Then, for each individual `produce` call:
* there are no headers (you can add headers using the `headers` kwargs parameter).
* the target partition is any (=-1) (you can set the target partition explicitly using the `partition` kwargs parameter).
* the timestamp is set automatically using the `CURRENT_TIME` setting (=0) (you can set the timestamp to a specfic value using the `timestamp` kwargs parameter).
* after producing each message, the producer does not call `flush` from the Kafka API (you can control this behavior using the `flush` kwargs parameter).

The call could also be written out as follows, assuming the values from the example configuration file `clusters/local.yaml`:


In [ ]:
c.produce_batch_size(1000)
c.flush_timeout(-1.0)
c.key_type("str")
c.value_type("json")
p = c.producer("topic_json", delivery_fun=None)
p.produce({"bla": 123}, key="123", headers=None, partition=-1, timestamp=0, flush=False)
p.produce({"bla": 456}, key="456", headers=None, partition=-1, timestamp=0, flush=False)
p.produce({"bla": 789}, key="789", headers=None, partition=-1, timestamp=0, flush=False)
p.close()


---

## More on Consuming Messages

Kafi uses the following defaults/assumptions here. For one, `cat` hides the implicit creation of a consumer object. Then, for setting up the consumer:
* It implicitly creates a new consumer group based on the current timestamp.
* `auto.offset.reset` is set to the corresponding value `auto.offset.reset` in the `kafi` section of the configuration file, e.g. `earliest` in `clusters/local.yaml`.
* `session.timeout.ms` is set to the corresponding value `session.timeout.ms` in the `kafi` section of the configuration file, e.g. `45000` milliseconds in `clusters/local.yaml`.
* `enable.auto.commit` is set to the corresponding value `enable.auto.commit` in the `kafi` section of the configuration file, e.g. `false` in `clusters/local.yaml`.
* The consume timeout is set to the corresponding value `consume.timeout` in the `kafi` section of the configuration file, e.g. `1.0` (for 1 second) in `clusters/local.yaml`. If you set this to -1, Kafi will "wait forever", as in a typical neverending consumer loop.
* The consumer group prefix is set to the corresponding value `consumer.group.prefix` in the `kafi` section of the configuration file, e.g. `""` in `clusters/local.yaml`.
* the maximum batch size for consuming is set to the corresponding value `consume.batch.size` in the `kafi` section of the configuration file, e.g. `1000` in `clusters/local.yaml`.
* The default key type is set to the corresponding value `key.type` in the `kafi` section of the configuration file, e.g. `str` in `clusters/local.yaml`. It can also be overridden with the `key_type` kwargs parameter.
* The default value type is set to the corresponding value `value.type` in the `kafi` section of the configuration file, e.g. `json` in `clusters/local.yaml`. It can also be overridden with the `value_type` kwargs parameter.

And for the `consume` calls:
* It attempts to read infinitely many messages (parameter `n=-1`)

The call could also be written out as follows, assuming that the current timestamp is `1732669768728` and the values from the example configuration file `clusters/local.yaml`:


In [ ]:
c.auto_offset_reset("earliest")
c.session_timeout_ms(45000)
c.enable_auto_commit(False)
c.consume_timeout(1.0)
c.consumer_group_prefix("")
c.consume_batch_size(1000)
c.key_type("str")
c.value_type("json")
co = c.consumer("topic_json", group="1732669768728")
co.consume(n=-1)
co.close()

---

## Architecture

### Storage

Essentially, Kafi is built on the concept of a "Storage". There are two kinds of Storages:
* Kafka (real Kafka: Kafka API or Kafka REST Proxy API)
* FS (file system: local file system, S3 or Azure Blob Storage)

The `Storage` class inherits from:
* `Shell`: Shell-like commands like `cat`, `head`, `tail`, `cp`...
* `Files`: Copying Kafka topics to files (`topic_to_file`) and vice versa (`file_to_topic`)
* `AddOns`: Higher-level add-on methods (`compact`, `compact_to`, `recreate`, `repeat`, `cp_groups_offsets`)
* `SchemaRegistry`: Schema Registry API

The classes `Shell`, `Files` and `AddOns` inherit from the class `Functional` which offers functional methods (`foldl`, `flatmap`, `map`, `filter`, `foreach`, `foldl_to`, `flatmap_to`, `map_to`, `filter_to`)

`Files` inherits indirectly from `Functional` through `Pandas` which allows to copy topics to Pandas dataframes (`topic_to_df`) and vice versa (`df_to_topic`).

```mermaid
---
title: Kafi class diagram (Storage)
---
classDiagram
    Functional <|-- Shell
    Functional <|-- AddOns
    Functional <|-- Pandas

    Pandas <|-- Files

    Shell <|-- Storage
    Files <|-- Storage
    AddOns <|-- Storage
    SchemaRegistry <|-- Storage

    Storage <|-- Kafka
    Kafka <|-- Cluster
    Kafka <|-- RestProxy

    Storage <|-- FS
    FS <|-- Local
    FS <|-- S3
    FS <|-- AzureBlob
```


### StorageConsumer

`StorageConsumer` is the base class for consuming records. It inherits from `Deserializer`, which in turn inherits from `SchemaRegistry`.

The individual storages have their own implementations:

```mermaid
---
title: Kafi class diagram (Consumer)
---
classDiagram
    SchemaRegistry <|-- Deserializer
    
    Deserializer <|-- Dechunker

    Dechunker <|-- StorageConsumer

    StorageConsumer <|-- KafkaConsumer
    KafkaConsumer <|-- ClusterConsumer
    KafkaConsumer <|-- RestProxyConsumer

    StorageConsumer <|-- FSConsumer
    FSConsumer <|-- LocalConsumer
    FSConsumer <|-- S3Consumer
    FSConsumer <|-- AzureBlobConsumer
```


### StorageProducer

`StorageProducer` is the base class for producing records. It inherits from `Serializer`, which in turn inherits from `SchemaRegistry`.

```mermaid
---
title: Kafi class diagram (Producer)
---
classDiagram
    SchemaRegistry <|-- Serializer

    Serializer <|-- StorageProducer

    StorageProducer <|-- KafkaProducer
    KafkaProducer <|-- ClusterProducer
    KafkaProducer <|-- RestProxyProducer

    StorageProducer <|-- FSProducer
    FSProducer <|-- LocalProducer
    FSProducer <|-- S3Producer
    FSProducer <|-- AzureBlobProducer
```


### StorageAdmin

`StorageAdmin` is the base class for administrative methods (e.g. Kafka Admin Client API). Implementations follow the same pattern:

```mermaid
---
title: Kafi class diagram (Admin)
---
classDiagram
    StorageAdmin <|-- KafkaAdmin
    KafkaAdmin <|-- ClusterAdmin
    KafkaAdmin <|-- RestProxyAdmin

    StorageAdmin <|-- FSAdmin
    FSAdmin <|-- LocalAdmin
    FSAdmin <|-- S3Admin
    FSAdmin <|-- AzureBlobAdmin
```
